# **OBJETIVOS DE ESTE CUADERNO**

1.  Utilizar SQLite como motor de base de datos dentro de colab para practicar operaciones SQL

2.  Realizar operaciones SQL de consulta sobre una tabla, N tablas y de agregación

<br/>

---

<br/>

## **NOTAS IMPORTANTES PARA USUARIOS EXPERIMENTADOS**

En estos ejercicios se crean tablas sin relaciones, sin claves foráneas y otros conceptos relevantes. Dado que el objetivo de este cuaderno es practicar sentencias SQL tipo "select" y operaciones de agregación, se omiten este tipo de temas para enfocarse en el objetivo principal.

Adicionalmente se utiliza sqlite como motor de bases de datos, primero por su simplicidad en cuanto a ejecutarse internamente en Colab, y segundo porque es suficiente para los objetivos de este cuaderno.



# **1. PREPARACIÓN DEL ENTORNO**

# **1.1 Importar librerías y módulos**

In [ ]:
##########
# PANDAS #
##########
import pandas as pd

##########
# SQLITE #
##########
import sqlite3 as sql

# **2. PREPARACIÓN DE LOS DATOS**

# **2.1 Base de datos "world": Obtener los datos de las tablas "country", "city" y "country_language" desde url públicas y llevarlos a DataFrames**

In [ ]:
url_country = 'https://raw.githubusercontent.com/navnebror/public_resources/refs/heads/main/databases/world2/country.csv'
url_city = 'https://raw.githubusercontent.com/navnebror/public_resources/refs/heads/main/databases/world2/city.csv'
url_language = 'https://raw.githubusercontent.com/navnebror/public_resources/refs/heads/main/databases/world2/country_language.csv'

df_country = pd.read_csv(url_country)
df_country.set_index('code', inplace=True)

df_city = pd.read_csv(url_city)
df_city.set_index('id', inplace=True)

df_language = pd.read_csv(url_language)
df_language.set_index(['country_code', 'language'], inplace=True)

# **2.2 Base de datos "chemistry": Obtener los datos de la tabla "elements" desde url públicas y llevarlos a DataFrames**

In [ ]:
url_elements = 'https://raw.githubusercontent.com/navnebror/public_resources/refs/heads/main/databases/chemistry/elements.json'

df_elements = pd.read_json(url_elements)
df_elements.set_index('atomic_number', inplace=True)

# **2.3 Base de datos "art": Obtener los datos de la tabla "singers" desde url públicas y llevarlos a DataFrames**

In [ ]:
url_singers = 'https://raw.githubusercontent.com/navnebror/public_resources/refs/heads/main/databases/art/singers.xml'

df_singers = pd.read_xml(url_singers)
df_singers.set_index('id', inplace=True)

# **2.4 Cargar los datos desde los DataFrame hacia una base de datos temporal llamada "test"**

In [ ]:
conn = sql.connect('/content/test.db')

conn.execute('drop table if exists country')
conn.execute('drop table if exists city')
conn.execute('drop table if exists country_language')
conn.execute('drop table if exists elements')
conn.execute('drop table if exists singers')

df_country.to_sql('country', conn)
df_city.to_sql('city', conn)
df_language.to_sql('country_language', conn)

df_elements.to_sql('elements', conn)

df_singers.to_sql('singers', conn)

# **3. EJERCICIOS RESUELTOS: CONSULTAS SQL BÁSICAS**

**NOTA: En estos ejercicios, los resultados siempre se llevan a un DataFrame**

# **3.1 Consultar los datos de todos los cantantes**

In [ ]:
sentencia_sql = """

select  *
from    singers

"""

df = pd.read_sql(sentencia_sql, conn)
df.set_index('id', inplace=True)

print(df)

# **3.2 Consultar los datos de todos los elementos químicos**

In [ ]:
sentencia_sql = """

select  *
from    elements

"""

df = pd.read_sql(sentencia_sql, conn)
df.set_index('name', inplace=True)

print(df)

# **3.3 Consultar los datos de todos los países**

In [ ]:
sentencia_sql = """

select  *
from    country

"""

df = pd.read_sql(sentencia_sql, conn)
df.set_index('code', inplace=True)

print(df)

# **3.4 Consultar los datos de todas las ciudades**

In [ ]:
sentencia_sql = """

select  *
from    city

"""

df = pd.read_sql(sentencia_sql, conn)
df.set_index('id', inplace=True)

print(df)

# **3.5 Consultar los datos de todos los idiomas**

In [ ]:
sentencia_sql = """

select  *
from    country_language

"""

df = pd.read_sql(sentencia_sql, conn)

print(df)

# **3.6 Consultar las ciudades de Noruega y Suiza**

In [ ]:
sentencia_sql = """

select  country.name,
        city.name
from    country,
        city
where   country.code = city.country_code
and     country.name IN ('Norway','Switzerland')

"""

df = pd.read_sql(sentencia_sql, conn)
print(df)

# **3.7 Consultar los idiomas oficiales y no oficiales que se hablan en Noruega y Suiza**

In [ ]:
sentencia_sql = """

select  country.name,
        country_language.language,
        country_language.is_official
from    country,
        country_language
where   country.code = country_language.country_code
and     country.name IN ('Norway','Switzerland')

"""

df = pd.read_sql(sentencia_sql, conn)
print(df)

# **4. EJERCICIOS PARA RESOLVER: CONSULTAS SQL SOBRE UNA TABLA**

En estos ejercicios, únicamente debe escribirse la sentencia SQL y ejecutar la celda. No debe cambiarse nada del código de Python

# **4.1 Mostrar nombre en inglés y en español, símbolo, número atómico, masa atómica, punto de fusión y punto de ebullición para los elementos con masa atómica entre 10 y 100**

In [ ]:
sentencia_sql = """

select name,
       name_spanish,
       symbol,
       atomic_number,
       atomic_mass,
       melting_point,
       boiling_point
from elements
where atomic_mass between 10 and 100

"""

df = pd.read_sql(sentencia_sql, conn)
print(df)

# **4.2 Mostrar nombre en inglés y en español, símbolo, número atómico, masa atómica, punto de fusión y punto de ebullición para los elementos con masa atómica entre 10 y 100 y con punto de ebullición menor a 0**

In [ ]:
sentencia_sql = """

select name,
       name_spanish,
       symbol,
       atomic_number,
       atomic_mass,
       melting_point,
       boiling_point
from elements
where atomic_mass between 10 and 100 and boiling_point < 0

"""

df = pd.read_sql(sentencia_sql, conn)
print(df)

# **4.3 Mostrar nombre en inglés y en español, símbolo y número atómico para los elementos con 4 órbitas (períodos)**

In [ ]:
sentencia_sql = """

select name,
       name_spanish,
       symbol,
       atomic_number
from elements
where period = 4

"""

df = pd.read_sql(sentencia_sql, conn)
print(df)

# **4.4 Mostrar nombre en español, símbolo, número atómico y configuración electrónica completa (no la reducida) para los elementos punto de fusión desconocido o punto de ebullición desconocido**

In [ ]:
sentencia_sql = """

select name_spanish,
       symbol,
       atomic_number,
       electronic_configuration
from elements
where melting_point is null or boiling_point is null

"""

df = pd.read_sql(sentencia_sql, conn)
print(df)

# **4.5 Mostrar el total de elementos por cantidad de órbitas (es decir, tantos elementos tienen tantas órbitas)**

In [ ]:
sentencia_sql = """

select period,
       count(*) as cantidad_elementos
from elements
group by period

"""

df = pd.read_sql(sentencia_sql, conn)
print(df)

# **4.6 Mostrar todos los datos disponibles para los elementos que comiencen con la letra H**

In [ ]:
sentencia_sql = """

select *
from elements
where name like 'H%'

"""

df = pd.read_sql(sentencia_sql, conn)
print(df)

# **4.7 Mostrar todos los datos disponibles para los elementos que comiencen con la letra H y que terminen con la letra O**

In [ ]:
sentencia_sql = """

select *
from elements
where name_spanish like 'H%O'

"""

df = pd.read_sql(sentencia_sql, conn)
print(df)

# **4.8 Mostrar nombre, apellido y fecha de nacimiento de los cantantes de POP**

In [ ]:
sentencia_sql = """

select  first_name,
        last_name,
        birth_date
from    singers
where   genre = 'POP'

"""

df = pd.read_sql(sentencia_sql, conn)
print(df)

# **4.9 Mostrar nombre, apellido y fecha de nacimiento de los cantantes de POP o de REGGAETON**

In [ ]:
sentencia_sql = """

select  first_name,
        last_name,
        birth_date
from    singers
where   genre in ('POP', 'REGGAETON')

"""

df = pd.read_sql(sentencia_sql, conn)
print(df)

# **4.10 Mostrar la cantidad total de premios que han ganado los cantantes de POP**

In [ ]:
sentencia_sql = """

SELECT SUM(awards) as total_premios
FROM singers
WHERE genre = 'POP';

"""

df = pd.read_sql(sentencia_sql, conn)
print(df)

# **4.11 Mostrar la cantidad total de premios que han ganado los cantantes por cada género musical**

In [ ]:
sentencia_sql = """

SELECT genre, SUM(awards) as total_premios_por_genero
FROM singers
group by genre

"""

df = pd.read_sql(sentencia_sql, conn)
print(df)

# **4.12 Mostrar cuantos cantantes son mujeres y cuantos hombres**

In [ ]:
sentencia_sql = """

select  gender,
        count(*) as cantidad_cantantes_por_genero
from    singers
group by gender

"""

df = pd.read_sql(sentencia_sql, conn)
print(df)

# **5. EJERCICIOS PARA RESOLVER: CONSULTAS SQL SOBRE VARIAS TABLAS**

En estos ejercicios, únicamente debe escribirse la sentencia SQL y ejecutar la celda. No debe cambiarse nada del código de Python

# **5.1 Mostrar nombre de ciudad, nombre de país y continente del país para los países de la región Caribe (en la base de datos se identifica como región "Caribbean")**

In [ ]:
sentencia_sql = """

select  city.name, country.name, country.continent
from city
join country on city.country_code = country.code
where region = 'Caribbean'

"""

df = pd.read_sql(sentencia_sql, conn)
print(df)

# **5.2 Mostrar nombre de país y nombre del idioma para los idiomas no oficiales que se hablan en américa del sur (en la base de datos se identifica como continente con nombre "South America")**

In [ ]:
sentencia_sql = """

select  country.name, country_language.language
from country
join country_language on country.code = country_language.country_code
where country.continent = 'South America' and country_language.is_official = 'F'

"""

df = pd.read_sql(sentencia_sql, conn)
print(df)

# **5.3 Mostrar cuantos países hay en cada continente**

In [ ]:
sentencia_sql = """

select  continent,
        count(*) as cantidad_paises
from    country
group by continent

"""

df = pd.read_sql(sentencia_sql, conn)
print(df)

# **5.4 Mostrar cuantos idiomas oficinales y no oficiales se hablan en cada país de Europa (en la base de datos se identifica como continente con nombre "Europe")**

In [ ]:
sentencia_sql = """

SELECT
    c.name AS pais,
    cl.is_official,
    COUNT(cl.language) AS cantidad_idiomas
FROM country_language cl
JOIN country c
    ON cl.country_code = c.code
WHERE c.continent = 'Europe'
GROUP BY c.name, cl.is_official
ORDER BY c.name;

"""

df = pd.read_sql(sentencia_sql, conn)
print(df)

# **5.5 Mostrar todos los datos de las ciudades de Noruega con más de 100.000 habitantes**

In [ ]:
sentencia_sql = """

select  city.*
from    city
join    country on city.country_code = country.code
where   country.name = 'Norway' and city.population > 100000

"""

df = pd.read_sql(sentencia_sql, conn)
print(df)

# **5.6 Mostrar nombre, continente y región de los países cuyo nombre finaliza con "istan"**

In [ ]:
sentencia_sql = """

select  name,
        continent,
        region
from    country
where   name like '%istan'

"""

df = pd.read_sql(sentencia_sql, conn)
print(df)

# **6. EJERCICIOS CON COMPLEJIDAD MAS ALTA**

NOTA: Estos ejercicios deben realizarse usando únicamente código SQL. Es decir, no "se vale" usar funciones de Pandas para calcular el resultado.

# **6.1 Mostrar el nombre del país más poblado del mundo, y su población**

In [ ]:
sentencia_sql = """

select  country.name,
        sum(city.population) as poblacion
from    country
join    city on country.code = city.country_code
group by country.name
order by poblacion desc
limit 1

"""

df = pd.read_sql(sentencia_sql, conn)
print(df)

# **6.2 Mostrar el nombre del país al cual pertenece la ciudad más poblada**

In [ ]:
sentencia_sql = """

select  country.name
from    country
join    city on country.code = city.country_code
where   city.population = (select max(population) from city)

"""

df = pd.read_sql(sentencia_sql, conn)
print(df)

# **6.3 Mostrar el porcentaje de la población que habla idiomas no oficiales en cada uno de los países de suramérica**

In [ ]:
sentencia_sql = """

SELECT
    c.name AS pais,
    SUM(CASE WHEN cl.is_official = 'F' THEN cl.percentage ELSE 0 END) AS porcentaje_no_oficial
FROM country c
JOIN country_language cl
    ON c.code = cl.country_code
WHERE c.continent = 'South America'
GROUP BY c.name
ORDER BY c.name;

"""

df = pd.read_sql(sentencia_sql, conn)
print(df)

# **6.4 Mostrar el nombre del idioma más hablado en el mundo**

In [ ]:
sentencia_sql = """

select  language,
        sum(percentage) as porcentaje
from    country_language
group by language
order by porcentaje desc
limit 1

"""

df = pd.read_sql(sentencia_sql, conn)
print(df)

# **6.5 Mostrar el nombre de las ciudades de Colombia, ordenadas de forma ascendente por su población**

In [ ]:
sentencia_sql = """

select  city.name
from    city
join    country on city.country_code = country.code
where   country.name = 'Colombia'
order by city.population

"""

df = pd.read_sql(sentencia_sql, conn)
print(df)